# Data Transformation
## HERD, GSS

This script takes files from the NSF HERD directory, filters to UNC's peer institutions using the `IPEDS_Institutions_Meta.csv` file, and outputs a dataframe readable by Tableau and PowerBI. Data in the output file an be updated simply by navigating to NSF HERD's public data file site (`https://ncses.nsf.gov/explore-data/microdata/higher-education-research-development`), downloading the latest file, and saving the `.csv` in the working directory `C:\Users\jmar18\OneDrive - University of North Carolina at Chapel Hill\Metrics PowerBI Conversion\NSF HERD`

## HERD


In [29]:
import pandas as pd
from os import listdir
from os.path import isfile, join

from pyprojroot import here 

wd = str(here('data/raw-data/NSF HERD/')) # finds the directory with data
pub = str(here('data/processed-data/')) # location where processed data is stored



In [30]:
# for some reason, a few institutions do not have their IPEDS UNITID loaded in the dataset. 
# I'm researching whether this has to do with the organizational structure. Perhaps this includes the Applied Physics Lab and that's why there's not unitid?
# For now, I want to be able to bring in Johns Hopkins, and others missing UNITIDs into my analysis. In this chunk, I'm hardcoding these values.

unique_id = {
    'Johns Hopkins University':'162928'
    ,'Embry-Riddle Aeronautical University':'133553'
    ,'Midwestern University':'143853'
    ,'University of New Hampshire':'183044'
    ,'Oregon State University':'209542'
    ,'Texas A&M University, College Station and Health Science Center':'228723'
    ,'University of Colorado Anschutz Medical Campus':'126562'
    ,'University of Colorado Denver':'126562'
    ,'University of Connecticut':'129020'
    ,'Bowling Green State University':'201441'
    ,'Kent State University':'203517'
    ,'Miami University':'204024'
    ,'Ohio State University, The':'204857'
    ,'Ohio University':'204796'
    ,'University of Cincinnati':'201885'
    #,'Desert Research Institute':''
    ,'Wright State University':'206604'
    ,'University of Akron, The':'200800'
    #,'University of Maryland Center for Environmental Science':''
    #,'Scripps Research Institute, The':''
    #,'Van Andel Research Institute':''
    ,'University of Mississippi':'176017'
    #,'City University of New York, The, Advanced Science Research Center':
    ,'City University of New York, Graduate School of Public Health and Health Policy':'190576'
    ,'University of Maryland':'163286'
    ,'University of Nebraska, Lincoln and Medical Center':'181464'
    ,'William & Mary':'231624'
    ,'Western Michigan University and Homer Stryker M.D. School of Medicine':'490373'
    ,'Southern University and A&M College, Agricultural Research and Extension Center':'160621'
    ,'Vanderbilt University and Vanderbilt University Medical Center':'221999'
}



In [48]:
# Pull in all files and concatenate
pre = pd.read_csv(wd+'\\'+files[-1]) # get column names for empty df
cols = pre.columns

data = pd.DataFrame(columns = cols)

for i in range(len(files)):
    temp = pd.read_csv(wd+'\\'+files[i], encoding='cp1252')
    data = pd.concat([data,temp])

df = data.reset_index().drop(columns='index')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xaa in position 14: invalid start byte

In [33]:
# maps values from the unique_id dictionary to the dataframe, creates a new unitid column
df['UNITID'] = df['inst_name_long'].map(unique_id).fillna(df['ipeds_unitid']).astype(str).str.split('.',expand=True)[0]

In [34]:
# Remove data associated to university, like name, location, etc. This will be joined in using IPEDS files

df = df[['inst_id','UNITID','year','questionnaire_no', 'question', 'row', 'column', 'data', 'status', 'othinfo', 'othinfo_s', 'standardized_agency_names']]
df['inst_id'] = df['inst_id'].dropna().astype(int) # format unitID to be filtered by peer list
#df = df[df['inst_id'].isin(peers)].reset_index().drop(columns='index')
#df = df.rename(columns={'ipeds_unitid':'UnitID'})
#df = df.rename(columns={'ipeds_unitid':'UnitID'})

In [35]:
# Get individual values for each institution. Create multiple two-column dataframes and merge together. 
#I'm not going to pivot here because some values need to be added toegether before pivoting

# Financial Data
fed_total = df[df['questionnaire_no']=='01.a'][['UNITID','year','row','data']].reset_index().drop(columns='index')
state_total = df[df['questionnaire_no']=='01.b'][['UNITID','year','row','data']].reset_index().drop(columns='index')
bus_total = df[df['questionnaire_no']=='01.c'][['UNITID','year','row','data']].reset_index().drop(columns='index')
nonprofit = df[df['questionnaire_no']=='01.d'][['UNITID','year','row','data']].reset_index().drop(columns='index')
inst_funds = df[df['questionnaire_no']=='01.e'][['UNITID','year','row','data']].reset_index().drop(columns='index')
all_other = df[df['questionnaire_no']=='01.g'][['UNITID','year','row','data']].reset_index().drop(columns='index')
total = df[df['questionnaire_no']=='01.g'][['UNITID','year','row','data']].reset_index().drop(columns='index')

# Personnel Data
res_headcount = df[(df['questionnaire_no']=='15') & (df['column']=='Researchers')][['UNITID','year','column','question','data']].reset_index().drop(columns='index')
res_headcount['row'] = res_headcount['column'] + '_' + res_headcount['question'].drop(columns=['column','question'])
res_headcount = res_headcount.drop(columns=['column','question'])

res_FTE = df[(df['questionnaire_no']=='16') & (df['column']=='Researchers')][['UNITID','year','column','question','data']].reset_index().drop(columns='index')
res_FTE['row'] = res_FTE['column'] + '_' + res_FTE['question'].drop(columns=['column','question'])
res_FTE = res_FTE.drop(columns=['column','question'])

total_headcount = df[(df['questionnaire_no']=='15') & (df['column']=='Total')][['UNITID','year','column','question','data']].reset_index().drop(columns='index')
total_headcount['row'] = total_headcount['column'] + '_' + total_headcount['question'].drop(columns=['column','question'])
total_headcount = total_headcount.drop(columns=['column','question'])

total_FTE = df[(df['questionnaire_no']=='16') & (df['column']=='Total')][['UNITID','year','column','question','data']].reset_index().drop(columns='index')
total_FTE['row'] = total_FTE['column'] + '_' + total_FTE['question'].drop(columns=['column','question'])
total_FTE = total_FTE.drop(columns=['column','question'])


In [36]:
data = pd.concat([fed_total,state_total,bus_total,nonprofit,inst_funds,all_other,total,res_headcount,res_FTE,total_headcount,total_FTE]).reset_index().drop(columns='index')
data = data.pivot_table(index=['UNITID','year'],columns='row',values='data')
data = data.reset_index()

In [40]:
# output NSF HERD Expenditures File
## Important note: HERD files from 2021 and earlier do not contain personnel data. This needs to be calculated using IPEDS data.
data.to_csv(pub + '\\nsf_herd_total_expenditures.csv')

## GSS

Download new data files: `https://ncses.nsf.gov/explore-data/microdata/graduate-students-postdoctorates-s-e`

| Variable | Description | Time Unit |
|----------|-------------|-----------|
| UNITID | Unique Identifier from IPEDS | Fall Term |
| pd_tot_all_races_v | Count of all post-docs | Fall Term |
| pd_tot_us_v | Count of post-docs who are US Citizens | Fall Term |
| pd_tot_forgn_v | Count of post-docs who are US Nonresidents | Fall Term |
| file | Generated from code, name of downloaded file from GSS site | Fall Term |


In [50]:
# GSS

wd = str(here('data/raw-data/NSF GSS/')) # finds the directory with data
pub = str(here('data/processed-data/')) # location where processed data is stored


In [51]:
# These files contain three sheets, Race, Support, and PD_NFR

pre = pd.read_excel(wd+'\\'+files[-1], sheet_name='PD_NFR') # get column names for empty df
cols = pre.columns

In [64]:
# Pull in all files and concatenate

data = pd.DataFrame(columns=['UNITID','pd_tot_all_races_v','pd_tot_us_v','dp_tot_forgn_v','file'])

for i in range(len(files)):
    temp = pd.read_excel(wd+'\\'+files[i], sheet_name='PD_NFR')[['UNITID','pd_tot_all_races_v','pd_tot_us_v','pd_tot_forgn_v']]
    temp['file'] = files[i]
    data = pd.concat([data,temp])
    print(files[i])
df = data.reset_index().drop(columns='index')

gss2017_Code.xlsx
gss2018_Code.xlsx
gss2019_Code.xlsx
gss2020_Code.xlsx
gss2021_Code.xlsx
gss2022_Code.xlsx
gss2023_Code.xlsx
gss2024_Code.xlsx


In [76]:
# for now, I'm getting the count of post-docs per institution. 
# This isn't a requirement, but I thought it would be interesting to see the breakdown by citizenship status
df = df.groupby(['UNITID','file']).sum().reset_index()
df['Fall Term'] = df['file'].str.split('_',expand=True)[0].str[-4:]

In [77]:
df

,UNITID,file,pd_tot_all_races_v,pd_tot_us_v,dp_tot_forgn_v,pd_tot_forgn_v,Fall Term
0,100654,gss2017_Code.xlsx,0,0,0,0.0,2017
1,100654,gss2018_Code.xlsx,0,0,0,0.0,2018
2,100654,gss2019_Code.xlsx,0,0,0,0.0,2019
3,100654,gss2020_Code.xlsx,0,0,0,0.0,2020
4,100654,gss2021_Code.xlsx,0,0,0,0.0,2021
...,...,...,...,...,...,...,...
5679,999999,gss2020_Code.xlsx,760,405,0,355.0,2020
5680,999999,gss2021_Code.xlsx,554,298,0,256.0,2021
5681,999999,gss2022_Code.xlsx,580,174,0,406.0,2022
5682,999999,gss2023_Code.xlsx,553,169,0,384.0,2023


In [79]:
df.to_csv(pub+'\\nsf_gss_postdocs.csv')